In [1]:
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score
)
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from xgboost import XGBClassifier

import mlflow

In [2]:
from pathlib import Path

DATA_PATH = Path("./Data")
RAW_DATA_PATH = DATA_PATH / "raw" / "Fraud_Data.csv"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Fraud Detection") 

<Experiment: artifact_location='file:C:/Users/trixr/Desktop/FraudDetection/mlflow-data/artifacts/1', creation_time=1781179797941, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781179797941, lifecycle_stage='active', name='Fraud Detection', tags={}, trace_location=None, workspace='default'>

In [4]:
import mlflow.sklearn 

mlflow.xgboost.autolog()
mlflow.lightgbm.autolog()
mlflow.sklearn.autolog()

# Load Data

In [5]:
X_train = pd.read_csv(PROCESSED_DATA_PATH / "4.1_X_train.csv")
y_train = pd.read_csv(PROCESSED_DATA_PATH / "4.2_y_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_PATH / "4.3_X_test.csv")
y_test = pd.read_csv(PROCESSED_DATA_PATH / "4.4_y_test.csv")

In [6]:
X_train.head()

,purchase_value,source,browser,sex,age,time_velocity,ip_user_share_count,device_user_share_count,day,hour,...,browser_FireFox,browser_IE,browser_Opera,browser_Safari,sex_F,sex_M,source_fr_enc,browser_fr_enc,source_targ_enc,browser_targ_enc
0,14,Ads,Chrome,F,38,7212744.0,0,0,25,11,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
1,14,Ads,Chrome,F,38,1.0,1,1,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
2,14,Ads,Chrome,F,38,1.0,2,2,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
3,14,Ads,Chrome,F,38,1.0,3,3,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798
4,14,Ads,Chrome,F,38,1.0,4,4,1,0,...,False,False,False,False,True,False,0.395388,0.406952,0.104356,0.111798


In [7]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120889 entries, 0 to 120888
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   purchase_value                120889 non-null  int64  
 1   source                        120889 non-null  object 
 2   browser                       120889 non-null  object 
 3   sex                           120889 non-null  object 
 4   age                           120889 non-null  int64  
 5   time_velocity                 120889 non-null  float64
 6   ip_user_share_count           120889 non-null  int64  
 7   device_user_share_count       120889 non-null  int64  
 8   day                           120889 non-null  int64  
 9   hour                          120889 non-null  int64  
 10  age_manual_binned             120889 non-null  int64  
 11  purchase_value_manual_binned  120889 non-null  int64  
 12  source_Ads                    120889 non-nul

In [9]:
features = [
    "time_velocity",
    "ip_user_share_count", "device_user_share_count",
    "day", "hour",
    "source_fr_enc", "browser_fr_enc",
    "sex_F", "sex_M",
    "age_manual_binned", "purchase_value_manual_binned"
]

X_train = X_train[features]
X_test = X_test[features]

In [10]:
from sklearn.utils.class_weight import compute_sample_weight


with mlflow.start_run(run_name="eval XGBoost with CV"):
    tscv = TimeSeriesSplit(n_splits=5)
    
    # "balanced" automatically penalizes the majority class and boosts the minority class
    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
    
    xgb = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )
    
    param_grid = {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.8, 1.0],
        "colsample_bytree": [0.7, 0.8, 1.0]
    }
    
    grid = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        cv=tscv,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=2
    )
    
    grid.fit(X_train, y_train, sample_weight=sample_weights)

2026/06/13 15:05:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 5 folds for each of 243 candidates, totalling 1215 fits


2026/06/13 15:16:00 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\trixr\Desktop\FraudDetection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/13 15:16:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution beca

🏃 View run enchanting-fowl-610 at: http://localhost:5000/#/experiments/1/runs/f600aa8233ff4f76bb5ebc29bb51d06c
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run wistful-worm-283 at: http://localhost:5000/#/experiments/1/runs/465166caa13840b1a8476e3fd4f5ee57
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run agreeable-rook-417 at: http://localhost:5000/#/experiments/1/runs/cec634a52b2d40e5b8067f52ead2509d
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run agreeable-mouse-633 at: http://localhost:5000/#/experiments/1/runs/b394562505f74eeabf7ae417eb6caacd
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run learned-bear-722 at: http://localhost:5000/#/experiments/1/runs/cc7adfc80b12444ab232c4ec58fdd90c
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run eval XGBoost with CV at: http://localhost:5000/#/experiments/1/runs/51d70fc3a2934750a06ac3b9db461ca2
🧪 View experiment at: http://localhost:5

In [11]:
best_params = grid.best_params_
mlflow.log_dict(best_params, "best_params.json")

In [15]:
(y_pred<0.05).astype(int)

array([1, 1, 1, ..., 1, 1, 1], shape=(30223,))

In [ ]:
thr = 0.005
while thr < 1:
    print(thr, f1_score(y_test, y_pred<thr))
    thr+=0.05

# Save CSV

In [ ]:
# X_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.1_X_train.csv", index=False)
# y_train_resampled.to_csv(PROCESSED_DATA_PATH / "5.2_y_train.csv", index=False)
